# Chapter 5 — Defending with Agents: Autonomous SecOps

## Objective

Assemble an autonomous, auditable SecOps pipeline — Planner-Executor-Verifier with a human gate, an event-bus audit trail, and signed events.

By the end you will be able to:

- Build a Planner → Executor → Verifier triage pipeline with a human gate
- Explain why an append-only log is a forensic audit trail
- Turn free-text alerts into structured tickets
- Emit signed, hash-chained audit events
- Apply the reversible-auto / irreversible-human-gate rule

## Summary

This notebook turns the previous attacks into defenses. You assemble an autonomous, auditable security-operations pipeline: a Planner assesses an alert, an Executor enriches and acts on it, and a Verifier validates the outcome and refuses to let an irreversible action through without a human. Around that pipeline you add an ordered event bus that doubles as a tamper-evident audit trail, a language-model helper that turns free-text alerts into structured tickets, and signed, hash-chained audit events. The guiding rule is bounded autonomy: reversible actions run at machine speed, irreversible ones stop for a person.

## Lab roadmap

**Chapter arc:** SecOps foundations → AI-based security processes → agentic SecOps workflows → AI SIEM, SOAR & security copilots

| Part | What you do | Time |
|---|---|---|
| Setup | Key (optional) | 2 min |
| 1 — Detect | Train a reconstruction scorer and turn its output into calibrated alerts | 10 min |
| 2 — Decide and act | Planner → Executor → Verifier with a tool belt and a human gate | 25 min |
| 3 — Wire it to a bus | Drive the pipeline from an append-only event log | 10 min |
| 4 — Free text in, ticket out | Regex vs. LLM extraction, with a schema check on the model's output | 15 min |
| 5 — Prove what happened | Signed, hash-chained audit events — and catch a tamperer | 15 min |
| Wrap-up | Quiz and takeaways | 10 min |

**Total:** about 1 hour 30 minutes. Look for the **🧪 Your turn** cells — those are the hands-on exercises, each with a self-check. **✅ Checkpoint** lines tell you what you should have seen before moving on.

## The big picture

Defense turns the previous attacks into an **autonomous, auditable SecOps pipeline**. Every SOC runs the same spine — collection → detection → triage → investigation → response — and AI augments each stage.

- **Planner → Executor → Verifier:** the Planner assesses severity, the Executor enriches and acts, the Verifier validates and gates.
- **Bounded autonomy:** reversible actions (block an IP, isolate an endpoint) run at machine speed; irreversible ones (wipe, delete, deprovision) stop for a human.
- **The audit trail:** an ordered, append-only event log is both the inter-agent bus and tamper-evident evidence; every action emits a signed event.
- **Measure it:** drive mean-time-to-detect and mean-time-to-respond down, and let every incident tune the next — the self-improving '10X SOC'.

**Planner - Executor - Verifier**

<svg viewBox="0 0 592 142" width="100%" style="max-width:592px;height:auto;font-family:system-ui,-apple-system,Segoe UI,Arial,sans-serif" xmlns="http://www.w3.org/2000/svg" role="img"><defs><marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#64748b"/></marker></defs><rect width="592" height="142" rx="12" fill="#ffffff"/><rect x="22" y="22" width="156" height="62" rx="9" fill="#1d4ed8"/><text x="100" y="45" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Planner</text><text x="100" y="63" fill="#e2e8f0" font-size="10.5" text-anchor="middle" dominant-baseline="middle">assess severity</text><line x1="178" y1="53" x2="218" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="218" y="22" width="156" height="62" rx="9" fill="#0f766e"/><text x="296" y="45" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Executor</text><text x="296" y="63" fill="#e2e8f0" font-size="10.5" text-anchor="middle" dominant-baseline="middle">enrich + act</text><line x1="374" y1="53" x2="414" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="414" y="22" width="156" height="62" rx="9" fill="#b91c1c"/><text x="492" y="45" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Verifier</text><text x="492" y="63" fill="#e2e8f0" font-size="10.5" text-anchor="middle" dominant-baseline="middle">validate / gate</text><text x="296" y="106" fill="#334155" font-size="12.5" text-anchor="middle">Reversible actions auto-run; irreversible ones stop at a human gate. Every step</text><text x="296" y="124" fill="#334155" font-size="12.5" text-anchor="middle">is a signed, append-only audit event.</text></svg>

## Setup

This notebook runs on **numpy** and **scikit-learn**. Production integrations — a message bus (**kafka**), threat-intel APIs, forensic tooling — are imported lazily or mocked, so every step has an offline path. The key wired below enables the live LLM extractor and playbook generator.

In [1]:
import os

# Load the OpenAI key into the environment (never printed): the key file on
# the lab VM, or OPENAI_API_KEY already set in the environment.
KEY_PATH = '/home/student/keys/key.txt'
client = None
if os.path.exists(KEY_PATH):
    os.environ["OPENAI_API_KEY"] = open(KEY_PATH).read().strip()
if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        client = OpenAI()
        print("OpenAI key loaded — client ready (key not shown).")
    except Exception as e:
        print("openai SDK unavailable; the live demo will be skipped:", e)
else:
    print(f"No key file at {KEY_PATH} and no OPENAI_API_KEY — the live demo will be skipped.")


No key file at /home/student/keys/key.txt and no OPENAI_API_KEY — the live demo will be skipped.


In [2]:
import json
import numpy as np

In [3]:
# Tiny helper the 🧪 exercises use to check your work.
def selfcheck(cond, ok_msg, hint):
    print(("\u2713 " + ok_msg) if cond else ("\u2717 Not yet \u2014 hint: " + hint))

---
# Part 1 — Detect

The pipeline starts with a detector raising an alert.

`ReconAnomalyScorer` is a portable stand-in for a trained autoencoder (Chapter 1): it learns a compressed representation of **benign** traffic with PCA and scores new traffic by how badly it reconstructs. Above the 99th-percentile benign error, it is an anomaly.

> **Security lens:** train on a benign baseline; a high reconstruction error means "far from normal" — the alert the Planner triages.

In [4]:
class ReconAnomalyScorer:
    """Stand-in for a trained VAE using PCA encode/decode for portability."""
    def __init__(self, n_components=6, pct=99):
        from sklearn.decomposition import PCA
        self.pca = PCA(n_components=n_components)
        self.pct = pct
        self.threshold = None

    def fit(self, benign):
        self.pca.fit(benign)
        self.threshold = np.percentile(self._err(benign), self.pct)
        return self

    def _err(self, X):
        Xr = self.pca.inverse_transform(self.pca.transform(X))
        return ((X - Xr) ** 2).mean(axis=1)

    def score(self, x):
        e = float(self._err(np.atleast_2d(x))[0])
        return {"score": e, "anomaly": bool(e > self.threshold)}


In [5]:
rng = np.random.default_rng(0)
scorer = ReconAnomalyScorer().fit(rng.normal(0, 1, size=(500, 16)))
benign_pt = rng.normal(0, 1, size=16)
attack_pt = rng.normal(3, 2, size=16)
print("anomaly (benign):", scorer.score(benign_pt))
print("anomaly (attack):", scorer.score(attack_pt))

# Calibrate: score = reconstruction error as a fraction of twice the
# anomaly threshold, capped at 0.99 — above-threshold traffic triages high.
def to_alert(host, pt):
    e = scorer.score(pt)["score"]
    return {"host": host, "score": round(float(min(0.99, e / (2 * scorer.threshold))), 2)}

det_alert  = to_alert("host-DB7", attack_pt)
det_alert2 = to_alert("host-W12", benign_pt)
print("\nalerts:", det_alert, det_alert2)

anomaly (benign): {'score': 0.3512349889913815, 'anomaly': False}
anomaly (attack): {'score': 5.393077802146983, 'anomaly': True}

alerts: {'host': 'host-DB7', 'score': 0.99} {'host': 'host-W12', 'score': 0.15}


---
# Part 2 — Decide and act: Planner → Executor → Verifier

The Planner assesses severity, the Executor enriches and acts, and the Verifier validates the outcome — rolling back any irreversible action a human has not approved.

> **Security lens:** the Verifier is the control point — policy checks, output validation, and the human gate on irreversible actions all live there. Which actions count as irreversible is **policy**, so it lives in one named set, `IRREVERSIBLE_ACTIONS`, not buried in code.

In [6]:
IRREVERSIBLE_ACTIONS = {"wipe", "delete"}

def planner(alert):
    sev = "high" if alert["score"] >= 0.8 else "low"
    return {"severity": sev,
            "action": "quarantine" if sev == "high" else "monitor"}

def executor(plan, run_tool=None):
    # The Executor enriches via its tool belt (VirusTotal / a CVE lookup /
    # firewall tools) before acting, and flags whether the action is
    # irreversible (so the Verifier can gate it).
    enrichment = run_tool(plan) if run_tool else None
    irreversible = plan["action"] in IRREVERSIBLE_ACTIONS
    return {"action": plan["action"], "done": True,
            "irreversible": irreversible, "enrichment": enrichment}

def verifier(result, human_ok=lambda r: True):
    # Human gate: an irreversible action with no approval rolls back.
    if result["irreversible"] and not human_ok(result):
        return {**result, "ok": False, "rolled_back": True}
    return {**result, "ok": result["done"], "rolled_back": False}

def pev_pipeline(alert, human_ok=lambda r: True, run_tool=None, plan_fn=None):
    plan = (plan_fn or planner)(alert)
    return verifier(executor(plan, run_tool), human_ok)

In [7]:
print("PEV (attack):", pev_pipeline(det_alert))
print("PEV (benign):", pev_pipeline(det_alert2))
print("PEV (irreversible, denied):",
      verifier({"action": "wipe", "done": True, "irreversible": True,
                "enrichment": None}, human_ok=lambda r: False))

PEV (attack): {'action': 'quarantine', 'done': True, 'irreversible': False, 'enrichment': None, 'ok': True, 'rolled_back': False}
PEV (benign): {'action': 'monitor', 'done': True, 'irreversible': False, 'enrichment': None, 'ok': True, 'rolled_back': False}
PEV (irreversible, denied): {'action': 'wipe', 'done': True, 'irreversible': True, 'enrichment': None, 'ok': False, 'rolled_back': True}


### 2.1 The Executor's tool belt

Before acting, the Executor enriches the alert with VirusTotal and CVE context — mocked here so it runs offline, with a bundled mini-database of real CVEs.

In [8]:
def vt_lookup(ioc):   return {"ioc": ioc, "malicious": 3, "vendors": 70,
                              "source": "VirusTotal (mock — no API key offline)"}

CVE_MINI_DB = {          # bundled, real CVEs so the lookup works offline
    "log4j":   {"cve": "CVE-2021-44228", "cvss": 10.0, "name": "Log4Shell"},
    "xz":      {"cve": "CVE-2024-3094",  "cvss": 10.0, "name": "xz-utils backdoor"},
    "openssl": {"cve": "CVE-2014-0160",  "cvss": 7.5,  "name": "Heartbleed"},
}
def cve_lookup(product):
    hit = CVE_MINI_DB.get(product.lower())
    return {"product": product,
            **(hit or {"cve": None, "note": "not in the bundled mini-DB"})}

def soc_tools(plan):     # the Executor's tool belt
    return {"vt":  vt_lookup("deadbeefcafe1234deadbeefcafe1234"),
            "cve": cve_lookup("log4j")}

decision = pev_pipeline(det_alert, run_tool=soc_tools)
print(json.dumps(decision, indent=2))

{
  "action": "quarantine",
  "done": true,
  "irreversible": false,
  "enrichment": {
    "vt": {
      "ioc": "deadbeefcafe1234deadbeefcafe1234",
      "malicious": 3,
      "vendors": 70,
      "source": "VirusTotal (mock \u2014 no API key offline)"
    },
    "cve": {
      "product": "log4j",
      "cve": "CVE-2021-44228",
      "cvss": 10.0,
      "name": "Log4Shell"
    }
  },
  "ok": true,
  "rolled_back": false
}


### 2.2 The human gate, made real

`human_ok=lambda r: False` is a stand-in. A real gate puts the request in an **approval queue** that a person works through, and the decision — including *who* approved — is recorded.

In [9]:
class ApprovalQueue:
    def __init__(self, decisions):
        self.decisions, self.log = decisions, []      # decisions: what the on-call human chose
    def __call__(self, result):
        approved, who = self.decisions.get(result["action"], (False, None))
        self.log.append({"action": result["action"], "approved": approved, "by": who})
        return approved

queue = ApprovalQueue({"wipe": (True, "alice@soc")})
for action in ["quarantine", "wipe", "delete"]:
    out = verifier(executor({"action": action}), human_ok=queue)
    print(f"{action:10} irreversible={out['irreversible']!s:5} ok={out['ok']!s:5} rolled_back={out['rolled_back']}")
print("approval log:", queue.log)

quarantine irreversible=False ok=True  rolled_back=False
wipe       irreversible=True  ok=True  rolled_back=False
delete     irreversible=True  ok=False rolled_back=True
approval log: [{'action': 'wipe', 'approved': True, 'by': 'alice@soc'}, {'action': 'delete', 'approved': False, 'by': None}]


> **✅ Checkpoint.** Quarantine never reaches the queue (reversible, runs at machine speed); `wipe` runs because alice approved it; `delete` is rolled back because nobody did. The approval log says who decided.

### 🧪 Your turn — extend the policy for a critical-vulnerability response

Add a planner rule: when the enrichment shows a CVE with **CVSS ≥ 9.0** on a high-severity alert, the plan is `reimage` (rebuild the host from a known-good image) instead of `quarantine`. Reimaging destroys evidence and state, so it must be gated: add it to `IRREVERSIBLE_ACTIONS`. Write `planner_v2(alert, enrichment)`.

In [10]:
def planner_v2(alert, enrichment):
    plan = planner(alert)
    # TODO: if plan is high severity and enrichment["cve"]["cvss"] >= 9.0 -> action "reimage"
    return plan

# TODO: make "reimage" an irreversible action

# ---- self-check (no need to edit below) ----
enr = soc_tools(None)
plan = planner_v2(det_alert, enr)
denied = verifier(executor(plan), human_ok=lambda r: False)
approved = verifier(executor(plan), human_ok=lambda r: True)
low = planner_v2(det_alert2, enr)
print("plan:", plan, "| without approval:", denied["rolled_back"], "| benign alert:", low["action"])
selfcheck(plan["action"] == "reimage" and denied["rolled_back"] and approved["ok"] and low["action"] == "monitor",
          "critical CVE -> reimage, and reimage waits for a human",
          "set plan['action'] = 'reimage' under the condition, and IRREVERSIBLE_ACTIONS.add('reimage')")

plan: {'severity': 'high', 'action': 'quarantine'} | without approval: False | benign alert: monitor
✗ Not yet — hint: set plan['action'] = 'reimage' under the condition, and IRREVERSIBLE_ACTIONS.add('reimage')


<details><summary><b>Show a solution</b> (try it yourself first)</summary>

Forgetting the second line is the classic failure: the planner gains a destructive action and the Verifier waves it through, because policy and capability were changed in different places.

```python
def planner_v2(alert, enrichment):
    plan = planner(alert)
    if plan["severity"] == "high" and (enrichment["cve"].get("cvss") or 0) >= 9.0:
        plan["action"] = "reimage"
    return plan

IRREVERSIBLE_ACTIONS.add("reimage")
```
</details>

---
# Part 3 — Wire it to a bus

In production the detector and the triage agent do not call each other; they communicate through a durable, **ordered** event bus. The append-only topic log is also the forensic record of everything that flowed through.

> **Security lens:** an ordered, append-only topic log doubles as a tamper-evident, replayable audit trail — the same log governance audits in Chapter 6.

`publish_alert` / `consume_alerts` are the Kafka versions; `LocalBus` is an in-process stand-in with the same shape, so the flow runs with no broker.

In [11]:
def publish_alert(alert, topic="alerts", servers="localhost:9092"):
    from kafka import KafkaProducer
    import json
    prod = KafkaProducer(bootstrap_servers=servers,
                         value_serializer=lambda v: json.dumps(v).encode())
    prod.send(topic, alert)        # every message is an ordered audit record
    prod.flush()

def consume_alerts(handler, topic="alerts", group="triage", servers="localhost:9092"):
    from kafka import KafkaConsumer
    import json
    for msg in KafkaConsumer(topic, group_id=group, bootstrap_servers=servers):
        handler(json.loads(msg.value))
class LocalBus:
    """In-process stand-in for the broker — same publish/consume shape over an
    append-only log, so the audit-trail flow runs with no service running.
    Swap in publish_alert/consume_alerts (Kafka) in production."""
    def __init__(self): self.topics = {}
    def publish(self, topic, event):
        self.topics.setdefault(topic, []).append(event)   # append-only, ordered
    def consume(self, topic, handler):
        for event in self.topics.get(topic, []):
            handler(event)


In [12]:
bus = LocalBus()

# Detector agent: publishes alerts.
for host, pt in [("host-DB7", attack_pt), ("host-W12", benign_pt), ("host-FS2", rng.normal(4, 2, 16))]:
    bus.publish("alerts", to_alert(host, pt))

# Triage agent: consumes alerts, runs PEV, publishes decisions to a second topic.
def triage_handler(alert):
    out = pev_pipeline(alert, run_tool=soc_tools)
    bus.publish("decisions", {"host": alert["host"], "action": out["action"], "ok": out["ok"]})

bus.consume("alerts", triage_handler)
for topic in ("alerts", "decisions"):
    print(f"{topic:9}:", bus.topics[topic])

alerts   : [{'host': 'host-DB7', 'score': 0.99}, {'host': 'host-W12', 'score': 0.15}, {'host': 'host-FS2', 'score': 0.99}]
decisions: [{'host': 'host-DB7', 'action': 'quarantine', 'ok': True}, {'host': 'host-W12', 'action': 'monitor', 'ok': True}, {'host': 'host-FS2', 'action': 'quarantine', 'ok': True}]


Replaying the `alerts` topic from the start reproduces every decision in order — that is what makes the log *evidence*. But an in-memory list (or a Kafka topic an admin can rewrite) is only as trustworthy as its storage. Part 5 makes the record tamper-*evident*.

---
# Part 4 — Free text in, ticket out

Analysts and users report incidents in free text. The Planner needs a structured ticket.

`extract_entities` is a regex extractor — fast, free, deterministic, and brittle. Watch what happens on messier reports.

> **Security lens:** free text in, structured ticket out — the contract that feeds the Planner and the event bus stays the same whether a regex or a model extracts.

In [13]:
import re
def extract_entities(text):
    host = re.search(r"\b(host[-\s]?\w+|\d+\.\d+\.\d+\.\d+)\b", text, re.I)
    sev  = re.search(r"\b(critical|high|medium|low)\b", text, re.I)
    return {
        "host":     host.group(0) if host else None,
        "severity": sev.group(0).title() if sev else None,
        "ioc":      re.findall(r"\b[a-f0-9]{32,64}\b", text, re.I),
    }

In [14]:
REPORTS = [
    "Critical alert on host-DB7, hash deadbeefcafe1234deadbeefcafe1234",
    "srv db-07 (10.0.4.17) looks really bad, sev is crit, saw sha256 9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08",
    "users on the 3rd floor say outlook is slow, probably nothing urgent",
]
for r in REPORTS:
    print(extract_entities(r))

{'host': 'host-DB7', 'severity': 'Critical', 'ioc': ['deadbeefcafe1234deadbeefcafe1234']}
{'host': '10.0.4.17', 'severity': None, 'ioc': ['9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08']}
{'host': None, 'severity': None, 'ioc': []}


The regex misses `crit` as a severity, and a model would handle all three. But a model's output is *untrusted* — it can omit fields, invent values, or be steered by an injected report. So the LLM extractor's output goes through `validate_ticket`, a deterministic schema check, before it may reach the Planner.

In [15]:
import re

def extract_entities_llm(text, client, model="gpt-4o-mini"):
    prompt = ("Extract fields from this security alert and return JSON with keys "
              "host, severity (Critical/High/Medium/Low), and ioc (list of hex hash "
              "strings). Alert:\n" + text)
    r = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}, temperature=0, max_tokens=200)
    return json.loads(r.choices[0].message.content)

SEVERITIES = {"Critical", "High", "Medium", "Low"}

def validate_ticket(t):
    problems = []
    if not isinstance(t.get("host"), str) or not t["host"].strip():
        problems.append("host missing")
    if t.get("severity") not in SEVERITIES:
        problems.append(f"severity {t.get('severity')!r} not in {sorted(SEVERITIES)}")
    if not isinstance(t.get("ioc"), list) or not all(
            isinstance(i, str) and re.fullmatch(r"[a-fA-F0-9]{32,64}", i) for i in t["ioc"]):
        problems.append("ioc must be a list of 32-64 hex chars")
    return (not problems), problems

for r in REPORTS:
    t = extract_entities_llm(r, client) if client else extract_entities(r)
    ok, problems = validate_ticket(t)
    print(f"{'LLM' if client else 'regex':5} {t}\n      -> {'to Planner' if ok else 'REJECTED: ' + '; '.join(problems)}")
print("\nhostile model output ->", validate_ticket({"host": "db7", "severity": "Urgent!!", "ioc": ["rm -rf /"]}))

regex {'host': 'host-DB7', 'severity': 'Critical', 'ioc': ['deadbeefcafe1234deadbeefcafe1234']}
      -> to Planner
regex {'host': '10.0.4.17', 'severity': None, 'ioc': ['9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08']}
      -> REJECTED: severity None not in ['Critical', 'High', 'Low', 'Medium']
regex {'host': None, 'severity': None, 'ioc': []}
      -> REJECTED: host missing; severity None not in ['Critical', 'High', 'Low', 'Medium']

hostile model output -> (False, ["severity 'Urgent!!' not in ['Critical', 'High', 'Low', 'Medium']", 'ioc must be a list of 32-64 hex chars'])


> **✅ Checkpoint.** Tickets that fail the schema never reach the Planner — whether the extractor was a brittle regex or a model that was confused or manipulated. This is the *deterministic output boundary* you will name in Chapter 6.

### 4.1 A GenAI SOAR playbook — with an allowlist

A model can draft the forensic playbook too. Its suggestions are checked against an allowlist of plugins the IR team actually runs; anything else is dropped, and a fixed, reviewed fallback is used when no model is wired.

In [16]:
ALLOWED_PLUGINS = {"windows.pslist", "windows.pstree", "windows.malfind", "windows.netscan",
                   "windows.cmdline", "windows.dlllist", "windows.handles", "windows.svcscan"}
FALLBACK_PLAYBOOK = [{"plugin": "windows.pslist",  "why": "enumerate processes"},
                     {"plugin": "windows.malfind", "why": "find injected code"},
                     {"plugin": "windows.netscan", "why": "surface C2 connections"}]

def gen_playbook(client=None, model="gpt-4o-mini"):
    if client is None:
        return FALLBACK_PLAYBOOK, []
    r = client.chat.completions.create(model=model, temperature=0, response_format={"type": "json_object"},
        messages=[{"role": "user", "content": "Return JSON {\"steps\": [...]} listing Volatility 3 plugins "
                   "to triage a Windows memory image; each step has 'plugin' and 'why'."}])
    steps = json.loads(r.choices[0].message.content).get("steps", [])
    kept = [s for s in steps if s.get("plugin") in ALLOWED_PLUGINS]
    dropped = [s.get("plugin") for s in steps if s.get("plugin") not in ALLOWED_PLUGINS]
    return (kept or FALLBACK_PLAYBOOK), dropped

playbook, dropped = gen_playbook(client)
for step in playbook:
    print(f"  {step['plugin']:18} {step['why']}")
print("dropped (not on the allowlist):", dropped or "none")

  windows.pslist     enumerate processes
  windows.malfind    find injected code
  windows.netscan    surface C2 connections
dropped (not on the allowlist): none


---
# Part 5 — Prove what happened: signed audit events

Every agent action emits a signed, hash-chained event before the next step runs: each event's `prev` field is the previous event's signature, so editing any event breaks every signature after it.

In [17]:
import hmac, hashlib

_AUDIT_KEY = b"lab-audit-key"        # in production: a per-agent key in a secrets manager / HSM
_chain = {"prev": "0" * 64}          # genesis: the first event chains to all-zeros

def _sig(ev):
    body = {k: v for k, v in ev.items() if k != "sig"}
    return hmac.new(_AUDIT_KEY, json.dumps(body, sort_keys=True).encode(), hashlib.sha256).hexdigest()

def sign_event(agent, action, outcome, ts="2026-01-01T00:00:00Z"):
    ev = {"agent": agent, "action": action, "outcome": outcome, "ts": ts, "prev": _chain["prev"]}
    ev["sig"] = _sig(ev)
    _chain["prev"] = ev["sig"]
    return ev

def run_signed(plan, human_ok=lambda r: True):
    result = executor(plan)
    evs = [sign_event("executor", plan["action"], "done" if result["done"] else "fail")]
    verdict = verifier(result, human_ok)
    evs.append(sign_event("verifier", "verify", "ok" if verdict["ok"] else "rolled_back"))
    return evs

plan = planner(det_alert)
events = [sign_event("planner", "plan", plan["action"])]
events += run_signed(plan)                                          # reversible -> auto-runs
events += run_signed({"action": "wipe"}, human_ok=lambda r: False)  # irreversible -> rolled back
for ev in events:
    print(f"{ev['agent']:9} {ev['action']:11} {ev['outcome']:11} prev={ev['prev'][:8]}.. sig={ev['sig'][:8]}..")

planner   plan        quarantine  prev=00000000.. sig=e741e260..
executor  quarantine  done        prev=e741e260.. sig=f5921f7f..
verifier  verify      ok          prev=f5921f7f.. sig=0ca67315..
executor  wipe        done        prev=0ca67315.. sig=10dbebd1..
verifier  verify      rolled_back prev=10dbebd1.. sig=c2c42213..


### 5.1 Verify the chain — and catch a tamperer

Checking that each `prev` matches the previous `sig` is not enough: an attacker who edits an event can also rewrite its `sig`. `verify_chain` **recomputes every HMAC** with the key, which the attacker does not have, and checks the links. It returns the index of the first bad event.

In [18]:
import copy

def verify_chain(evs):
    prev = "0" * 64
    for i, ev in enumerate(evs):
        if ev["prev"] != prev:
            return False, i, "broken link (event removed, reordered, or inserted)"
        if not hmac.compare_digest(ev["sig"], _sig(ev)):
            return False, i, "bad signature (event edited)"
        prev = ev["sig"]
    return True, None, "intact"

print("original          :", verify_chain(events))

# An insider edits the record to hide the rolled-back wipe.
edited = copy.deepcopy(events)
edited[4]["outcome"] = "ok"
print("outcome edited    :", verify_chain(edited))

# A smarter insider also recomputes a SHA-256 'signature' — but without the key.
forged = copy.deepcopy(edited)
forged[4]["sig"] = hashlib.sha256(json.dumps(forged[4], sort_keys=True).encode()).hexdigest()
print("sig forged w/o key:", verify_chain(forged))

# Or deletes the inconvenient events from the middle...
print("middle deleted    :", verify_chain(events[:2] + events[3:]))
# ...or cuts the tail off the log.
print("tail truncated    :", verify_chain(events[:3]))

original          : (True, None, 'intact')
outcome edited    : (False, 4, 'bad signature (event edited)')
sig forged w/o key: (False, 4, 'bad signature (event edited)')
middle deleted    : (False, 2, 'broken link (event removed, reordered, or inserted)')
tail truncated    : (True, None, 'intact')


> **✅ Checkpoint.** Edits, forgeries, and deletions are caught and located — except the last case: a log cut off at the **end** is still internally consistent. Anchoring the latest signature somewhere the attacker cannot write (a separate log service, a WORM bucket) closes that gap.

## Quiz

Answer each question, then fill in the `answers` dict in the next cell (`"A"`, `"B"`, …) and run it to grade yourself.

**1. The Verifier receives an irreversible action with no human approval. What happens?**

- A. It runs, and is logged
- B. It is retried later
- C. It is rolled back
- D. It is downgraded to monitor

**2. In the exercise, why must 'reimage' be added to IRREVERSIBLE_ACTIONS as well as to the planner?**

- A. Otherwise the Verifier treats it as reversible and runs it without a human
- B. Otherwise the planner cannot choose it
- C. Otherwise the bus drops the event
- D. It does not need to be

**3. An LLM extractor returns severity 'Urgent!!'. What should happen?**

- A. Map it to Critical
- B. Pass it to the Planner — the model knows best
- C. Reject the ticket at the schema check before it reaches the Planner
- D. Retry until it says Critical

**4. An insider edits an audit event and recomputes a plain SHA-256 over it. Why does verify_chain still catch it?**

- A. SHA-256 is broken
- B. The timestamp changed
- C. The bus keeps a copy
- D. The signature is an HMAC; without the key the forged value does not match

**5. Which tampering does the hash chain alone NOT detect?**

- A. Editing a middle event
- B. Deleting middle events
- C. Truncating events from the end of the log
- D. Reordering events

**6. Why does the playbook generator check plugins against ALLOWED_PLUGINS?**

- A. To save tokens
- B. So a confused or manipulated model cannot add steps the IR team never vetted
- C. Volatility requires it
- D. To sort the steps

In [19]:
import base64
_KEY = {'q1': 'Qw==', 'q2': 'QQ==', 'q3': 'Qw==', 'q4': 'RA==', 'q5': 'Qw==', 'q6': 'Qg=='}   # base64 so the answers aren't obvious at a glance
_WHY = {'q1': 'Irreversible + unapproved = rolled back; reversible actions run at machine speed.', 'q2': 'Capability and policy must change together, or a new destructive action skips the gate.', 'q3': 'Model output is untrusted input; a deterministic schema check guards the Planner.', 'q4': 'Keyed signatures can only be produced by the key holder.', 'q5': 'A truncated chain is internally consistent; anchor the latest signature externally.', 'q6': 'An allowlist is a deterministic boundary on what model output can cause.'}

def check(answers):
    key = {q: base64.b64decode(v).decode() for q, v in _KEY.items()}
    if all(str(answers.get(q, '')).strip().upper() in ('', '?') for q in key):
        print('Fill in the answers dict below (A/B/C/D) and re-run to grade.')
        return
    score = 0
    for q in sorted(key, key=lambda k: int(k[1:])):
        got = str(answers.get(q, '')).strip().upper()
        if got == key[q]:
            score += 1; print(f'\u2713 {q}: {got}')
        else:
            print(f'\u2717 {q}: you put {got or "\u2014"} \u2014 correct is {key[q]}. {_WHY[q]}')
    print(f'\nScore: {score}/{len(key)}')

In [20]:
answers = {
    "q1": "?",   # A / B / C / D
    "q2": "?",   # A / B / C / D
    "q3": "?",   # A / B / C / D
    "q4": "?",   # A / B / C / D
    "q5": "?",   # A / B / C / D
    "q6": "?",   # A / B / C / D
}
check(answers)

Fill in the answers dict below (A/B/C/D) and re-run to grade.


## Key takeaways

- Defense is a Planner → Executor → Verifier pipeline: reversible actions auto-run, irreversible ones stop for a recorded human decision.
- Policy (what is irreversible) lives in one place; when you give an agent a new capability, update the policy in the same change.
- An ordered, append-only event log is both the inter-agent bus and the forensic record.
- Model output is untrusted input: schema checks and allowlists sit between the model and anything that acts.
- Signed, hash-chained events make the trail evidence: verification recomputes the HMACs and pinpoints tampering — anchor the head of the chain externally.

**Up next:** Chapter 6 turns these controls into governance: a scored checklist, a RACI, and a Zero-Trust audit that can block go-live.